## 📧 Email Spam Classification

Given *data about emails*, let's try to predict whether a given email will be **spam** or not.

We will use a TensorFlow/Keras neural network with word embeddings to make our predictions.

Data source: https://www.kaggle.com/datasets/chandramoulinaidu/spam-classification-for-basic-nlp

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import tensorflow as tf

In [2]:
data = pd.read_csv('archive/Spam Email raw text for NLP.csv')
data

,CATEGORY,MESSAGE,FILE_NAME
0,1,"Dear Homeowner,\n\n \n\nInterest Rates are at ...",00249.5f45607c1bffe89f60ba1ec9f878039a
1,1,ATTENTION: This is a MUST for ALL Computer Use...,00373.ebe8670ac56b04125c25100a36ab0510
2,1,This is a multi-part message in MIME format.\n...,00214.1367039e50dc6b7adb0f2aa8aba83216
3,1,IMPORTANT INFORMATION:\n\n\n\nThe new domain n...,00210.050ffd105bd4e006771ee63cabc59978
4,1,This is the bottom line. If you can GIVE AWAY...,00033.9babb58d9298daa2963d4f514193d7d6
...,...,...,...
5791,0,"I'm one of the 30,000 but it's not working ver...",00609.dd49926ce94a1ea328cce9b62825bc97
5792,0,Damien Morton quoted:\n\n>W3C approves HTML 4 ...,00957.e0b56b117f3ec5f85e432a9d2a47801f
5793,0,"On Mon, 2002-07-22 at 06:50, che wrote:\n\n\n\...",01127.841233b48eceb74a825417d8d918abf8
5794,0,"Once upon a time, Manfred wrote :\n\n\n\n> I w...",01178.5c977dff972cd6eef64d4173b90307f0


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5796 entries, 0 to 5795
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CATEGORY   5796 non-null   int64
 1   MESSAGE    5796 non-null   str  
 2   FILE_NAME  5796 non-null   str  
dtypes: int64(1), str(2)
memory usage: 136.0 KB


### Preprocessing

In [4]:
df = data.copy()

In [5]:
# Drop FILE_NAME column
df = df.drop('FILE_NAME', axis=1)

In [6]:
df

,CATEGORY,MESSAGE
0,1,"Dear Homeowner,\n\n \n\nInterest Rates are at ..."
1,1,ATTENTION: This is a MUST for ALL Computer Use...
2,1,This is a multi-part message in MIME format.\n...
3,1,IMPORTANT INFORMATION:\n\n\n\nThe new domain n...
4,1,This is the bottom line. If you can GIVE AWAY...
...,...,...
5791,0,"I'm one of the 30,000 but it's not working ver..."
5792,0,Damien Morton quoted:\n\n>W3C approves HTML 4 ...
5793,0,"On Mon, 2002-07-22 at 06:50, che wrote:\n\n\n\..."
5794,0,"Once upon a time, Manfred wrote :\n\n\n\n> I w..."


In [7]:
# Split df into X and y
y = df['CATEGORY'].copy()
X = df['MESSAGE'].copy()

In [8]:
y

0       1
1       1
2       1
3       1
4       1
       ..
5791    0
5792    0
5793    0
5794    0
5795    0
Name: CATEGORY, Length: 5796, dtype: int64

In [9]:
X

0       Dear Homeowner,\n\n \n\nInterest Rates are at ...
1       ATTENTION: This is a MUST for ALL Computer Use...
2       This is a multi-part message in MIME format.\n...
3       IMPORTANT INFORMATION:\n\n\n\nThe new domain n...
4       This is the bottom line.  If you can GIVE AWAY...
                              ...                        
5791    I'm one of the 30,000 but it's not working ver...
5792    Damien Morton quoted:\n\n>W3C approves HTML 4 ...
5793    On Mon, 2002-07-22 at 06:50, che wrote:\n\n\n\...
5794    Once upon a time, Manfred wrote :\n\n\n\n> I w...
5795    If you run Pick, and then use the "New FTOC" b...
Name: MESSAGE, Length: 5796, dtype: str

In [16]:
# Create the tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=30000)

In [17]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

In [18]:
X_train

4530    I don't know for skimming that article which D...
4478    I'm talking specifically about the last ~24 ho...
1156    <html>\n\n<body>\n\n<p align=3D"center"><br>\n...
5539    \n\nSo are PNGs still kosh?\n\n\n\n\n\n\n\n\n\...
2088    right Mike,\n\n\n\ni will agree to disagree bu...
                              ...                        
905     <html>\n\n\n\n<body>\n\n\n\n<font size="2" PTS...
5192    \n\n\n\nformail did the trick. Thanks to those...
3980    URL: http://www.askbjoernhansen.com/archives/2...
235     <html>\n\n<head>\n\n   <meta http-equiv=3D"Con...
5157    >>>>> "E" == Elias Sinderson <elias@cse.ucsc.e...
Name: MESSAGE, Length: 4057, dtype: str

In [19]:
# Fit the tokenizer
tokenizer.fit_on_texts(X_train)

In [20]:
len(tokenizer.word_index)

79082

In [27]:
# Convert texts to sequences
seq = tokenizer.texts_to_sequences(X_train)

In [31]:
def get_sequences(texts, tokenizer, train=True, max_seq_length=None):
    sequences = tokenizer.texts_to_sequences(texts)
    if train == True:
        max_seq_length = np.max(list(map(lambda x: len(x), sequences)))
    sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_seq_length, padding='post')
    return sequences

In [33]:
X_train = get_sequences(X_train, tokenizer, train=True)
X_test = get_sequences(X_test, tokenizer, train=False, max_seq_length=X_train.shape[1])

In [35]:
X_train.shape

(4057, 14804)

In [36]:
X_test.shape

(1739, 14804)

In [37]:
y_train.value_counts()

CATEGORY
0    2738
1    1319
Name: count, dtype: int64

### Training

In [40]:
inputs = tf.keras.Input(shape=(14804, ))

embedding = tf.keras.layers.Embedding(
    input_dim = 30000,
    output_dim = 64
)(inputs)

flatten = tf.keras.layers.Flatten()(embedding)

In [41]:
inputs

<KerasTensor shape=(None, 14804), dtype=float32, sparse=False, ragged=False, name=keras_tensor_2>

In [42]:
embedding

<KerasTensor shape=(None, 14804, 64), dtype=float32, sparse=False, ragged=False, name=keras_tensor_3>

In [43]:
flatten

<KerasTensor shape=(None, 947456), dtype=float32, sparse=False, ragged=False, name=keras_tensor_4>

In [44]:
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(flatten)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [45]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [46]:
print(model.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 14804)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_1 (Embedding)              │ (None, 14804, 64)           │       1,920,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 947456)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │         947,457 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,867,457 (10.94 MB)

 Trainable params: 2,867,457 (10.94 MB)

 Non-trainable params: 0 (0.00 B)

None


In [47]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    batch_size = 32,
    epochs = 100,
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
    ]
)

Epoch 1/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 173ms/step - accuracy: 0.7596 - auc: 0.6938 - loss: 10.2802 - val_accuracy: 0.9778 - val_auc: 0.9923 - val_loss: 0.1360
Epoch 2/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 165ms/step - accuracy: 0.9843 - auc: 0.9981 - loss: 0.0696 - val_accuracy: 0.9865 - val_auc: 0.9979 - val_loss: 0.0521
Epoch 3/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 164ms/step - accuracy: 0.9960 - auc: 0.9995 - loss: 0.0259 - val_accuracy: 0.9877 - val_auc: 0.9989 - val_loss: 0.0409
Epoch 4/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 21s 166ms/step - accuracy: 0.9985 - auc: 1.0000 - loss: 0.0119 - val_accuracy: 0.9901 - val_auc: 0.9991 - val_loss: 0.0324
Epoch 5/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 21s 166ms/step - accuracy: 0.9988 - auc: 1.0000 - loss: 0.0070 - val_accuracy: 0.9889 - val_auc: 0.9992 - val_loss: 0.0311
Epoch 6/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 21s 166ms/step - accuracy: 0.9997 - auc: 1.0000 - loss: 0.0043 - val_accuracy: 0.9901 - val_auc: 0.9976 - val_loss: 0.0290
Epoch 7/100
102

### Results

In [48]:
results = model.evaluate(X_test, y_test, verbose=0)

In [51]:
print("    Test Loss: {:.4f}".format(results[0]))
print("Test Accuracy: {:.2f}%".format(results[1]*100))
print("     Test AUC: {:.4f}".format(results[2]))

    Test Loss: 0.0308
Test Accuracy: 99.14%
     Test AUC: 0.9985


In [60]:
y_pred = np.squeeze(np.array(model.predict(X_test) >= 0.5, dtype=int))
y_pred

55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step


array([1, 1, 0, ..., 0, 0, 1], shape=(1739,))

In [61]:
(y_pred != y_test).sum()

np.int64(15)

In [62]:
len(y_pred)

1739

In [63]:
(y_pred != y_test).sum() / len(y_pred)

np.float64(0.008625646923519264)